# WIE3007 Individual Assignment
**Qistina Aqilah binti Abdul Razak Tuah (23001946)**  
Domain: Private healthcare (KlinikSihat) | Target: `no_show` | Seed: 001946

In [1]:
# Setup: seed from matric number (last six digits = 001946)
SEED = 1946

import random
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

DATA_DIR = '../data/'

# Part B — Data Preparation, Quality and Missing Data

## B1. Generate the dataset

**Design.** Each row is one scheduled appointment at a KlinikSihat clinic — the same grain as `fact_appointment` in Part A, flattened (denormalised) so that the dimension attributes (clinic, doctor, patient, channel, date, time slot) sit next to the measures. The target class is **`no_show`** (Yes/No).

**Realistic relationships built in** (through a logistic model of no-show probability):
- longer booking lead time → more no-shows;
- appointments with a reminder (especially two reminders) → fewer no-shows;
- patients with past no-shows → more no-shows;
- younger adults (18–30) and self-pay patients → more no-shows; patients aged 60+ and panel patients → fewer;
- longer distance to the clinic → more no-shows;
- Monday and early-morning (before 10:00) appointments → more no-shows; locum doctors → slightly more.

Other attributes are linked realistically too: older patients book more by phone and younger ones by app; the consultation fee depends on the consultation type and doctor type; health screenings are mostly for older patients.

All randomness comes from `rng`, seeded with `SEED = 1946` (matric 23001946 → last six digits 001946), so the notebook produces exactly the same data on every run.

In [ ]:
# ---------- B1.1 Reference data (dimension-like lookups) ----------
N_APPOINTMENTS = 1000          # brief: at least 500 rows
N_PATIENTS     = 700           # some patients have more than one appointment
N_DOCTORS      = 25

clinics = pd.DataFrame({
    "clinic_code": [f"KS{i:02d}" for i in range(1, 11)],
    "clinic_area": ["Petaling Jaya", "Shah Alam", "Subang Jaya", "Klang", "Kajang",
                    "Cheras", "Kepong", "Bangsar", "Putrajaya", "Rawang"],
    "clinic_state": ["Selangor"] * 5 + ["Kuala Lumpur"] * 3 + ["Putrajaya", "Selangor"],
})

doctors = pd.DataFrame({
    "doctor_id": [f"D{i:03d}" for i in range(1, N_DOCTORS + 1)],
    "clinic_code": rng.choice(clinics["clinic_code"], size=N_DOCTORS),
    "doctor_type": rng.choice(["Permanent", "Locum"], size=N_DOCTORS, p=[0.8, 0.2]),
})
# make sure every clinic has at least one doctor
doctors.loc[:9, "clinic_code"] = clinics["clinic_code"].values

# Patients: age drives insurance type, preferred channel and past no-shows
age = np.clip(np.round(rng.gamma(shape=6.0, scale=6.5, size=N_PATIENTS)), 1, 90).astype(int)
insurance = np.where(
    (age >= 22) & (age <= 58),
    rng.choice(["Panel", "Insurance", "Self-pay"], size=N_PATIENTS, p=[0.50, 0.20, 0.30]),
    rng.choice(["Panel", "Insurance", "Self-pay"], size=N_PATIENTS, p=[0.05, 0.25, 0.70]),
)
lam = np.where(age < 30, 0.8, np.where(age < 60, 0.4, 0.2))   # younger patients: more past no-shows
patients = pd.DataFrame({
    "patient_id": [f"P{i:05d}" for i in range(1, N_PATIENTS + 1)],
    "patient_age": age,
    "gender": rng.choice(["F", "M"], size=N_PATIENTS, p=[0.54, 0.46]),
    "insurance_type": insurance,
    "distance_km": np.round(np.clip(rng.lognormal(mean=np.log(6), sigma=0.6, size=N_PATIENTS), 0.5, 45), 1),
    "prior_no_shows_12m": np.minimum(rng.poisson(lam), 5),
})
print(clinics.shape, doctors.shape, patients.shape)
patients.head()

In [ ]:
# ---------- B1.2 Generate the appointments ----------
n = N_APPOINTMENTS
appt = pd.DataFrame({"record_id": np.arange(1, n + 1)})
appt["appointment_no"] = [f"APT-{i:06d}" for i in appt["record_id"]]

# patient and doctor for each appointment
appt["patient_id"] = rng.choice(patients["patient_id"], size=n)
appt["doctor_id"] = rng.choice(doctors["doctor_id"], size=n)
appt = appt.merge(patients, on="patient_id", how="left").merge(doctors, on="doctor_id", how="left")
appt = appt.merge(clinics, on="clinic_code", how="left")

# appointment date: Monday-Saturday, January-June 2026 (clinics closed on Sunday)
all_days = pd.date_range("2026-01-05", "2026-06-27", freq="D")
open_days = all_days[all_days.day_of_week < 6]
appt["appointment_date"] = rng.choice(open_days, size=n)

# booking lead time: about 30% book 0-2 days ahead, the rest further ahead
short = rng.integers(0, 3, size=n)
long_ = np.minimum(np.round(rng.exponential(scale=10, size=n)) + 3, 60).astype(int)
appt["lead_time_days"] = np.where(rng.random(n) < 0.30, short, long_)
appt["booking_date"] = appt["appointment_date"] - pd.to_timedelta(appt["lead_time_days"], unit="D")
appt["day_of_week"] = appt["appointment_date"].dt.day_name()

# time slot: 15-minute slots from 09:00 to 21:45
slots = pd.date_range("09:00", "21:45", freq="15min").strftime("%H:%M")
appt["time_slot"] = rng.choice(slots, size=n)
hour = appt["time_slot"].str[:2].astype(int)
appt["session"] = np.select([hour < 12, hour < 17], ["Morning", "Afternoon"], default="Evening")

# booking channel depends on age: older patients phone, younger use the app
p_young = [0.45, 0.30, 0.15, 0.10]   # App, WhatsApp, Phone, Website
p_mid   = [0.30, 0.35, 0.20, 0.15]
p_old   = [0.10, 0.25, 0.55, 0.10]
channels = ["App", "WhatsApp", "Phone", "Website"]
appt["booking_channel"] = [
    rng.choice(channels, p=(p_young if a < 30 else p_mid if a < 60 else p_old))
    for a in appt["patient_age"]
]

# consultation type depends on age (screenings mostly for older patients)
appt["consultation_type"] = [
    rng.choice(["General", "Follow-up", "Health screening"],
               p=([0.70, 0.25, 0.05] if a < 40 else [0.50, 0.30, 0.20]))
    for a in appt["patient_age"]
]

# reminders: none for same-day bookings; digital channels get more reminders
digital = appt["booking_channel"].isin(["App", "WhatsApp"]).to_numpy()
p_rem = np.where(digital, 0.85, 0.55)
first = (rng.random(n) < p_rem) & (appt["lead_time_days"].to_numpy() >= 1)
second = first & (rng.random(n) < 0.45) & (appt["lead_time_days"].to_numpy() >= 3)
appt["reminders_sent"] = first.astype(int) + second.astype(int)

# consultation fee (RM) depends on consultation type and doctor type, rounded to RM 0.50
base_fee = appt["consultation_type"].map({"General": 45, "Follow-up": 35, "Health screening": 150})
locum_extra = np.where(appt["doctor_type"] == "Locum", 10, 0)
fee = base_fee + locum_extra + rng.normal(0, 5, size=n)
appt["consultation_fee_rm"] = np.round(fee * 2) / 2

# ---------- target: no_show via a logistic model ----------
a = appt["patient_age"].to_numpy()
logit = (
    -2.55
    + 0.060 * np.minimum(appt["lead_time_days"], 30)
    - 0.80 * (appt["reminders_sent"] >= 1)
    - 0.40 * (appt["reminders_sent"] == 2)
    + 0.55 * np.minimum(appt["prior_no_shows_12m"], 3)
    + 0.60 * ((a >= 18) & (a <= 30))
    - 0.50 * (a >= 60)
    + 0.45 * (appt["insurance_type"] == "Self-pay")
    - 0.35 * (appt["insurance_type"] == "Panel")
    + 0.035 * appt["distance_km"]
    + 0.35 * (appt["day_of_week"] == "Monday")
    + 0.40 * (hour < 10)
    + 0.25 * (appt["doctor_type"] == "Locum")
    - 0.60 * (appt["consultation_type"] == "Health screening")
)
p_no_show = 1 / (1 + np.exp(-logit.to_numpy(dtype=float)))
appt["no_show"] = np.where(rng.random(n) < p_no_show, "Yes", "No")

# final column order (dates as ISO strings, as they would be exported from the source system)
cols = ["record_id", "appointment_no", "patient_id", "patient_age", "gender", "insurance_type",
        "distance_km", "prior_no_shows_12m", "clinic_code", "clinic_area", "clinic_state",
        "doctor_id", "doctor_type", "booking_date", "appointment_date", "lead_time_days",
        "day_of_week", "time_slot", "session", "booking_channel", "consultation_type",
        "reminders_sent", "consultation_fee_rm", "no_show"]
ground_truth = appt[cols].copy()
ground_truth["booking_date"] = ground_truth["booking_date"].dt.strftime("%Y-%m-%d")
ground_truth["appointment_date"] = ground_truth["appointment_date"].dt.strftime("%Y-%m-%d")
print(ground_truth.shape)
ground_truth.head()

In [ ]:
# ---------- B1.3 Sanity checks: size, target balance and built-in relationships ----------
gt = ground_truth
print(f"Rows: {len(gt)} | Attributes (excluding record_id and target): {gt.shape[1] - 2}")
print(f"Numeric: {gt.select_dtypes('number').columns.drop('record_id').tolist()}")
print(f"Categorical: {gt.select_dtypes(exclude='number').columns.drop(['no_show']).tolist()}")
print(f"\nNo-show rate: {(gt['no_show'] == 'Yes').mean():.1%}\n")

rate = lambda s: (s == "Yes").mean().round(3)
checks = {
    "lead_time_band": pd.cut(gt["lead_time_days"], [-1, 2, 14, 60], labels=["0-2 days", "3-14 days", "15+ days"]),
    "reminders_sent": gt["reminders_sent"],
    "prior_no_shows_12m": gt["prior_no_shows_12m"].clip(upper=3),
    "insurance_type": gt["insurance_type"],
    "age_band": pd.cut(gt["patient_age"], [0, 17, 30, 59, 90], labels=["0-17", "18-30", "31-59", "60+"]),
}
for name, grp in checks.items():
    print(f"No-show rate by {name}:")
    print(gt.groupby(grp, observed=True)["no_show"].agg(rate).to_string(), "\n")

In [ ]:
# ---------- B1.4 Save the ground truth (kept untouched for B4 evaluation) ----------
ground_truth.to_csv(DATA_DIR + "ground_truth.csv", index=False)
print("Saved", DATA_DIR + "ground_truth.csv", ground_truth.shape)

### Injecting data-quality problems into a copy → `raw.csv`

| Problem | Column(s) | How it is injected | Requirement met |
|---|---|---|---|
| Missing values — **MAR** | `patient_age` | Missing far more often when `booking_channel` = Phone (front-desk staff skip the date of birth) than for other channels; about 8% overall | 5–10% missing; missingness depends on another observed column |
| Missing values — **MCAR** | `consultation_fee_rm` | 7% of rows chosen completely at random (billing interface outage) | 5–10% missing; unrelated to any data |
| Duplicate rows | all | 15 rows copied exactly and appended, then rows shuffled | ≥ 10 duplicates |
| Inconsistent formats | `appointment_date`, `booking_date` | about 20% written as DD/MM/YYYY instead of YYYY-MM-DD | inconsistent date formats |
| | `clinic_state` | mixed case, extra spaces and abbreviations (e.g. `SELANGOR`, ` Selangor `, `KL`, `W.P. Kuala Lumpur`) | upper/lower case states |
| | `gender` | `F`/`M` mixed with `Female`/`Male` and lower case | inconsistent categories |
| Out-of-range values | `patient_age` | 4 impossible ages (−3, −1, 150, 212) | ≥ 5 out-of-range values in total (10 here) |
| | `consultation_fee_rm` | 3 fees with the sign flipped (negative) | |
| | `lead_time_days` | 3 negative lead times | |

All injected cells are recorded by `record_id` in `injection_log` so the cleaning in B3 and the evaluation in B4 can be checked against them.

In [ ]:
# ---------- B1.5 Inject problems into a copy ----------
raw = ground_truth.copy()
n = len(raw)
injection_log = {}

# (1) MAR missingness in patient_age: depends on booking_channel (Phone much higher)
p_miss_age = np.where(raw["booking_channel"] == "Phone", 0.28, 0.03)
mar_rows = raw.index[rng.random(n) < p_miss_age]
raw.loc[mar_rows, "patient_age"] = np.nan
injection_log["MAR_patient_age"] = raw.loc[mar_rows, "record_id"].tolist()

# (2) MCAR missingness in consultation_fee_rm: 7% of rows completely at random
mcar_rows = raw.index[rng.permutation(n)[: int(round(0.07 * n))]]
raw.loc[mcar_rows, "consultation_fee_rm"] = np.nan
injection_log["MCAR_consultation_fee_rm"] = sorted(raw.loc[mcar_rows, "record_id"].tolist())

# (3) Out-of-range values (chosen only from cells that are not already missing)
ok_age = raw.index[raw["patient_age"].notna()]
age_rows = rng.choice(ok_age, size=4, replace=False)
raw.loc[age_rows, "patient_age"] = [-3, -1, 150, 212]
injection_log["OOR_patient_age"] = sorted(raw.loc[age_rows, "record_id"].tolist())

ok_fee = raw.index[raw["consultation_fee_rm"].notna()]
fee_rows = rng.choice(ok_fee, size=3, replace=False)
raw.loc[fee_rows, "consultation_fee_rm"] = -raw.loc[fee_rows, "consultation_fee_rm"]
injection_log["OOR_consultation_fee_rm"] = sorted(raw.loc[fee_rows, "record_id"].tolist())

lead_rows = rng.choice(raw.index[raw["lead_time_days"] > 0], size=3, replace=False)
raw.loc[lead_rows, "lead_time_days"] = -raw.loc[lead_rows, "lead_time_days"]
injection_log["OOR_lead_time_days"] = sorted(raw.loc[lead_rows, "record_id"].tolist())

# (4) Inconsistent date formats: ~20% of rows written as DD/MM/YYYY
date_rows = raw.index[rng.random(n) < 0.20]
for col in ["appointment_date", "booking_date"]:
    raw.loc[date_rows, col] = pd.to_datetime(raw.loc[date_rows, col]).dt.strftime("%d/%m/%Y")
injection_log["FMT_dates_ddmmyyyy"] = raw.loc[date_rows, "record_id"].tolist()

# (5) Inconsistent state names: case, spaces and abbreviations
variants = {
    "Selangor": ["SELANGOR", "selangor", " Selangor ", "Selangor "],
    "Kuala Lumpur": ["KL", "kuala lumpur", "W.P. Kuala Lumpur", "KUALA LUMPUR"],
    "Putrajaya": ["PUTRAJAYA", "putrajaya", "W.P. Putrajaya"],
}
state_rows = raw.index[rng.random(n) < 0.25]
raw.loc[state_rows, "clinic_state"] = [rng.choice(variants[s]) for s in raw.loc[state_rows, "clinic_state"]]
injection_log["FMT_clinic_state"] = raw.loc[state_rows, "record_id"].tolist()

# (6) Inconsistent gender labels
gender_rows = raw.index[rng.random(n) < 0.15]
gmap = {"F": ["Female", "female", "f"], "M": ["Male", "male", "m"]}
raw.loc[gender_rows, "gender"] = [rng.choice(gmap[g]) for g in raw.loc[gender_rows, "gender"]]
injection_log["FMT_gender"] = raw.loc[gender_rows, "record_id"].tolist()

# (7) Exact duplicate rows: copy 15 rows, append, then shuffle the whole table
dup_src = rng.choice(raw.index, size=15, replace=False)
injection_log["DUP_source_record_ids"] = sorted(raw.loc[dup_src, "record_id"].tolist())
raw = pd.concat([raw, raw.loc[dup_src]], ignore_index=True)
raw = raw.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(raw.shape)
print({k: len(v) for k, v in injection_log.items()})
raw.head()

In [ ]:
# ---------- B1.6 Verify every injection meets the brief ----------
base = raw.drop_duplicates()            # the duplicates are exact copies, so drop them for rate checks
print(f"raw.csv rows: {len(raw)} (= {len(ground_truth)} + {len(raw) - len(ground_truth)} duplicates)")
print(f"Exact duplicate rows: {raw.duplicated().sum()}\n")

for col, kind in [("patient_age", "MAR"), ("consultation_fee_rm", "MCAR")]:
    pct = base[col].isna().mean()
    print(f"{col:22s} {kind:4s} missing: {base[col].isna().sum():3d} rows = {pct:.1%}  "
          f"{'OK (5-10%)' if 0.05 <= pct <= 0.10 else 'CHECK'}")

print("\nMissing patient_age by booking_channel (MAR evidence):")
print(base.groupby("booking_channel")["patient_age"].apply(lambda s: s.isna().mean()).round(3).to_string())

oor = ((base["patient_age"] < 0) | (base["patient_age"] > 120)).sum() \
      + (base["consultation_fee_rm"] < 0).sum() + (base["lead_time_days"] < 0).sum()
print(f"\nOut-of-range values: {oor}  {'OK (>= 5)' if oor >= 5 else 'CHECK'}")
print(f"Dates in DD/MM/YYYY: {base['appointment_date'].str.contains('/').sum()}")
print(f"clinic_state labels: {sorted(map(str, base['clinic_state'].unique()))}")
print(f"gender labels: {sorted(map(str, base['gender'].unique()))}")

In [ ]:
# ---------- B1.7 Save raw.csv ----------
raw.to_csv(DATA_DIR + "raw.csv", index=False)
print("Saved", DATA_DIR + "raw.csv", raw.shape)

## B2. Profile the data

## B3. Clean and standardise

## B4. Impute and evaluate

# Part C — Decision Tree Using ID3

## C4. Verify with code (scikit-learn)

## C5. Apply to own data

# Part D — Association Rule Mining Using Apriori

## D3. Verify with mlxtend